# Geolocation GeoGuessr AI: Cuarto Modelo CNN (Versión Optimizada)
Este notebook recupera y mejora la estrategia de entrenamiento del tercer modelo. Mantenemos la resolución 224x224 pero incluimos **Swish activation**, **SpatialDropout2D**, **Label Smoothing (0.1)** y recortamos (capping max_weight=5.0) en los pesajes de clase (Class Weights).

## 1. Importación de Librerías y Configuración

In [ ]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print(f'TensorFlow Version: {tf.__version__}')
print(f'Num GPUs Available: {len(tf.config.list_physical_devices("GPU"))}')

## 2. Definición del Dataset

In [ ]:
current_dir = pathlib.Path(os.getcwd())
data_dir = current_dir / 'data' / 'compressed_dataset' # Ruta local por defecto

print(f"Buscando imágenes en: {data_dir}")

filepaths = list(data_dir.glob(r'**/*.jpg'))
labels = [os.path.split(os.path.split(x)[0])[1] for x in filepaths]

df = pd.DataFrame({'Filepath': [str(x) for x in filepaths], 'Label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Filtrar países con pocas imágenes
MIN_IMAGENES = 50
num_fotos_por_pais = df['Label'].value_counts()
paises_validos = num_fotos_por_pais[num_fotos_por_pais >= MIN_IMAGENES].index
df_filtrado = df[df['Label'].isin(paises_validos)].copy()

print(f"Hemos mantenido {len(paises_validos)} países de {len(num_fotos_por_pais)}")
print(f"Dataset filtrado: {len(df_filtrado)} imágenes")

train_df, val_df = train_test_split(
    df_filtrado,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df_filtrado['Label']
)

print(f"Imágenes de Entrenamiento: {len(train_df)}")
print(f"Imágenes de Validación: {len(val_df)}")

## 3. Data Generators y Capping de Pesos

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

datagen = ImageDataGenerator(rescale=1./255)

train_images = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_images = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

num_classes = len(train_images.class_indices)

# Calcular pesos de clase balanceados limitando su valor para prevenir saltos de gradiente
train_labels = train_images.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

max_weight = 5.0
class_weights = np.clip(class_weights, a_min=None, a_max=max_weight)
class_weight_dict = dict(enumerate(class_weights))

## 4. Construcción de la CNN Personalizada

In [ ]:
from tensorflow.keras.regularizers import l2

def residual_block(x, filters, kernel_size=3, stride=1):
    shortcut = x

    x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    x = layers.Conv2D(filters, kernel_size, strides=1, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('swish')(x)
    
    x = layers.SpatialDropout2D(0.1)(x)
    return x

def build_custom_cnn(num_classes):
    inputs = keras.Input(shape=(224, 224, 3))
    
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.15)(x)
    x = layers.RandomZoom(0.2)(x)
    x = layers.RandomContrast(0.25)(x)
    
    x = layers.Conv2D(32, (7, 7), strides=2, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2, padding='same')(x)
    
    x = residual_block(x, filters=64, stride=1)
    x = residual_block(x, filters=64, stride=1)
    
    x = residual_block(x, filters=128, stride=2)
    x = residual_block(x, filters=128, stride=1)
    
    x = residual_block(x, filters=256, stride=2)
    x = residual_block(x, filters=256, stride=1)
    
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(1024, activation='swish', kernel_regularizer=l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(512, activation='swish', kernel_regularizer=l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model

model = build_custom_cnn(num_classes)
model.summary()


## 5. Compilación y Callback

In [ ]:
from tensorflow.keras.optimizers.schedules import CosineDecay

EPOCHS = 50
initial_learning_rate = 0.001
decay_steps = int((len(train_images)) * EPOCHS)

lr_schedule = CosineDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=decay_steps,
    alpha=0.01
)

opt = keras.optimizers.Adam(learning_rate=lr_schedule)
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer=opt,
    loss=loss_fn,
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
)

callbacks_list = [
    callbacks.ModelCheckpoint(
        'best_cuarto_cnn.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]


## 6. Entrenamiento

In [ ]:
history = model.fit(
    train_images,
    validation_data=val_images,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    class_weight=class_weight_dict
)

## 7. Gráficas de Rendimiento

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Progreso de la Precisión')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Progreso de la Pérdida (Loss)')

plt.show()